# 🤖 Step 3 — Train Fraud Detection Model

We train a LightGBM classifier to predict fraud.
The model learns patterns from labeled historical transactions.

In [ ]:
# Import MLflow and scikit-learn libraries for training and evaluating the fraud model
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Load the Silver fraud features into pandas so scikit-learn can train on them
df = spark.table('silver_fraud_features').toPandas()

# Choose the feature columns that capture transaction amount, timing, velocity, and anomaly patterns
features = ['Amount', 'TimeSinceLastTxnMins', 'NumTxnLast24h',
            'AvgTxnAmount30d', 'location_jump', 'high_velocity',
            'amount_spike', 'fast_repeat']
target = 'IsFraud'

# Split the dataset into model inputs and the binary fraud label
# Use pd.to_numeric to safely handle Categorical dtype from Spark's toPandas()
X = df[features].apply(pd.to_numeric, errors='coerce').fillna(0)
y = pd.to_numeric(df[target], errors='coerce').fillna(0).astype(int)

# Create training and test sets so model performance can be checked on unseen transactions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Print the training and testing sample counts for a quick sanity check
print(f'Training on {len(X_train)} samples, testing on {len(X_test)}')

In [ ]:
# Start an MLflow run so metrics and the trained fraud model are tracked together
with mlflow.start_run(run_name='fraud_detection_v1'):
    # Train a Gradient Boosting classifier on the engineered fraud features
    model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Predict fraud labels for the holdout test set and calculate accuracy
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Log the evaluation metric and trained model artifact to MLflow for reuse in scoring
    mlflow.log_metric('accuracy', accuracy)
    mlflow.sklearn.log_model(model, 'fraud_model')
    
    # Print the overall accuracy and class-level precision and recall summary
    print(f'\n✅ Model Accuracy: {accuracy:.1%}')
    print('\nDetailed Report:')
    print(classification_report(y_test, y_pred, target_names=['Legitimate','Fraud']))